# Optimizing Agent Prompts with the NeMo Agent Toolkit Optimizer

This notebook builds on `01_nat_basics.ipynb` and walks through the toolkit's built-in
[optimizer](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/main/docs/source/improve-workflows/optimizer.md),
which can automatically search for better hyperparameters *and* better prompts for a workflow.

**How it works**

- Numeric fields (like `temperature`, `top_p`) are tuned with [Optuna](https://optuna.org/) — a standard
  hyperparameter search library.
- Prompt fields are tuned with a custom **genetic algorithm (GA)**: a population of prompt variants is scored on a
  dataset, the best performers are kept/combined ("crossover"), and an LLM is used to "mutate" prompts into new
  variants — repeated over several generations.

In this notebook we'll optimize a *prompt* only (GA), not any numeric hyperparameters, to keep things fast and
focused. Specifically, we'll tune the `additional_instructions` field of a `react_agent` — a field the toolkit
already marks as optimizable in its source (an `OptimizableField`) — to improve how accurately the agent answers a
small set of general-knowledge questions.

**What we'll do**
1. Install the optimizer + evaluation extras
2. Create a tiny evaluation dataset
3. Write a workflow config with an `eval:` section (for scoring) and an `optimizer:` section (for tuning)
4. Measure baseline accuracy with `nat eval`
5. Run `nat optimize` and inspect the winning prompt
6. Re-measure accuracy with the optimized prompt and compare

## 0. Setup

In [1]:
# Only needed if you didn't install requirements.txt into this kernel's environment already.
# - langchain: provides the react_agent workflow type and the built-in prompt mutation/crossover functions
# - eval: provides `nat eval` and the ragas-based evaluators
# - config-optimizer: provides `nat optimize` (Optuna + genetic algorithm)
# %pip install -q "nvidia-nat[langchain,eval,config-optimizer]==1.8.0"

In [2]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA_API_KEY (from build.nvidia.com): ")
else:
    print("NVIDIA_API_KEY already set.")

NVIDIA_API_KEY already set.


## 1. Create a small evaluation dataset

`nat eval` (and the optimizer, which uses `nat eval` internally) score a workflow against a dataset of
`question` / `answer` pairs. Each `question` is sent to the workflow as input, and the workflow's
`generated_answer` is compared against the ground-truth `answer` by an evaluator.

We keep this dataset intentionally tiny (6 items) so the whole notebook runs in a few minutes. For a real project,
use a larger, more representative dataset.

Two of the six questions (ids 3 and 5) are classic ["cognitive reflection test"](https://en.wikipedia.org/wiki/Cognitive_Reflection_Test)
questions — a bat-and-ball price puzzle and a machines/widgets rate puzzle. Both have an intuitive-but-wrong answer
(10 cents; 100 minutes) that pattern-matching produces instantly, and a correct answer (5 cents; 5 minutes) that
only shows up if the model actually works through the reasoning instead of pattern-matching the first answer that
comes to mind. That makes them a good fit for *this* notebook specifically: a plain-trivia dataset tends to hit a
100% ceiling immediately (nothing for the optimizer to improve), whereas these need the kind of behavioral nudge —
"think it through, don't just state the first answer" — that prompt engineering can actually supply, unlike pure
factual gaps that no amount of prompting fixes.

In [23]:
%%writefile eval_dataset.json
[
  {"id": "1", "question": "What is the capital of France?", "answer": "Paris"},
  {"id": "2", "question": "What is the chemical symbol for gold?", "answer": "Au"},
  {"id": "3", "question": "How many r's are there in the word 'strawberry'?", "answer": "3"},
  {"id": "4", "question": "Who wrote the book Never Split the Difference?", "answer": "Chris Voss"},
  {"id": "5", "question": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?", "answer": "5 minutes"},
  {"id": "6", "question": "What is the largest planet in our solar system?", "answer": "Jupiter"}
]

Overwriting eval_dataset.json


## 2. Write the workflow + eval + optimizer config

A few things are new compared to notebook 1:

- **`workflow.optimizable_params: [additional_instructions]`** — turns on optimization for the `react_agent`'s
  `additional_instructions` field. This field already carries an `OptimizableField` definition in the toolkit's
  source, so we don't need to write any custom Python config classes ourselves.
- **`functions.prompt_mutator` / `functions.prompt_crossover`** — the built-in `prompt_init` and `prompt_recombiner`
  function types. These are LLM-powered "mutation" and "crossover" operators the GA calls each generation to
  produce new prompt candidates. `system_objective` tells the mutating LLM what a *good* instruction looks like
  for our task.
- **`eval:`** — a dataset (the file we just wrote) plus a `tunable_rag_evaluator`, which uses a judge LLM to score
  the generated answer against the expected answer (0 to 1) with a customizable judge prompt. We use
  `default_scoring: false` with a custom `judge_llm_prompt` — that swaps the evaluator's built-in 4-field
  `{coverage_score, correctness_score, relevance_score, reasoning}` schema for a simpler 2-field
  `{score, reasoning}` one. Fewer required fields means less that can go wrong when the judge LLM formats its
  response, which matters because a parsing failure here is silently scored `0.0` — see the "parsing failures"
  note in step 3 below for why that's worth watching for. (The `ragas` evaluator shown in some upstream docs isn't
  part of this pinned `nvidia-nat==1.8.0` release — `nat info components -t evaluator` shows what's actually
  available in your installed version.)
- **`optimizer:`** — `numeric.enabled: false` because we're only tuning the prompt here; `prompt.enabled: true`
  with a small population/generation count so the demo finishes quickly (increase these for real use, see the
  wrap-up section).

In [50]:
%%writefile optimizer_config.yml
functions:
  current_datetime:
    _type: current_datetime
  prompt_mutator:
    _type: prompt_init
    optimizer_llm: nim_llm
    optimizer_prompt: >
      You are an expert at optimizing prompts for LLMs. Your task is to take a given prompt and
      suggest an optimized version of it. Note that the prompt might be a template with variables
      and curly braces. Remember to always keep the variables and curly braces in the prompt the
      same. Only modify the instructions in the prompt that are not variables. CRITICAL FORMATTING
      RULE: never introduce any new curly brace characters, code blocks, backticks, or JSON
      anywhere in your output. Respond only in plain English prose sentences, with no markdown
      formatting, lists, or examples of any kind. The system is meant to achieve the following
      objective
      {system_objective}
      Of which, the prompt is one part. The details of the prompt and context as below.
    system_objective: >
      Write short, plain-language instructions (2-3 sentences, prose only, no lists, no code, no
      JSON, no curly braces) that help a ReAct agent give direct, factually correct final answers
      to general-knowledge questions, without unnecessary hedging or verbosity.
  prompt_crossover:
    _type: prompt_recombiner
    optimizer_llm: nim_llm
    optimizer_prompt: >
      You are an expert at combining prompt instructions for LLMs. Your task is to merge two
      prompts for the same objective into a single, stronger prompt. Do not introduce new
      variables or modify existing placeholders. CRITICAL FORMATTING RULE: never introduce any new
      curly brace characters, code blocks, backticks, or JSON anywhere in your output. Respond
      only in plain English prose sentences, with no markdown formatting, lists, or examples of
      any kind. The system is meant to achieve the following objective
      {system_objective}
    system_objective: >
      Combine the strongest guidance from two candidate instruction sets into a single concise,
      plain-language instruction (prose only, no lists, no code, no JSON, no curly braces).

llms:
  # base_url is driven by an optional NVIDIA_BASE_URL environment variable — see the README.
  # nemotron-3-nano is the model used successfully throughout notebooks 1 and 3 — reliably
  # produces well-formed ReAct-style tool calls. Very small models (e.g. 1B-class) have been
  # observed to garble the "Action: <tool_name>" line badly enough that the agent never
  # recovers and returns an empty final answer — see the note in step 3 below.
  nim_llm:
    _type: nim
    model_name: nvidia/meta/llama-3.2-1b-instruct
    max_tokens: 500
    temperature: 1.0
    api_key: ${NVIDIA_API_KEY}
    base_url: ${NVIDIA_BASE_URL}
  nim_judge_llm:
    _type: nim
    model_name: openai/openai/gpt-5.5
    temperature: 1.
    max_tokens: 500
    api_key: ${NVIDIA_API_KEY}
    base_url: ${NVIDIA_BASE_URL}

workflow:
  _type: react_agent
  tool_names: [current_datetime]
  llm_name: nim_llm
  verbose: false
  max_retries: 5
  parse_agent_response_max_retries: 3
  optimizable_params: [additional_instructions]

eval:
  general:
    output_dir: ./eval_baseline
    dataset:
      _type: json
      file_path: eval_dataset.json
  evaluators:
    accuracy:
      _type: tunable_rag_evaluator
      llm_name: nim_judge_llm
      # default_scoring: false uses a simpler 2-field {score, reasoning} schema instead of the
      # 4-field {coverage_score, correctness_score, relevance_score, reasoning} one. Fewer
      # required fields means less surface area for the judge LLM to get wrong, which matters
      # because a parsing failure here is silently scored 0.0 (indistinguishable from "wrong
      # answer") — see the diagnostic cells below that flag these explicitly.
      default_scoring: false
      judge_llm_prompt: >
        Score how well the generated answer matches the expected answer for this
        general-knowledge question, on a scale from 0.0 (completely wrong or irrelevant) to 1.0
        (fully correct and directly answers the question). If the generated answer is empty,
        blank, or missing, you MUST score it 0.0 and state in your reasoning that no answer was
        provided — never assume, guess, or infer what the answer might have been just because you
        know the expected answer. Respond with ONLY a single raw JSON object with keys "score" and
        "reasoning" — no markdown code fences, no backticks, and no commentary or preamble outside
        the JSON object.

optimizer:
  output_path: optimizer_results
  numeric:
    enabled: false
  prompt:
    enabled: true
    prompt_population_init_function: prompt_mutator
    prompt_recombination_function: prompt_crossover
    ga_population_size: 3
    ga_generations: 2
    ga_crossover_rate: 0.7
    ga_mutation_rate: 0.3
    ga_elitism: 1
    ga_selection_method: tournament
    ga_tournament_size: 3
    ga_parallel_evaluations: 2
  reps_per_param_set: 1
  eval_metrics:
    correctness:
      evaluator_name: accuracy
      direction: maximize
      weight: 1.0

Overwriting optimizer_config.yml


## 3. Baseline: score the un-optimized workflow

Before optimizing, let's see how the agent does with its default `additional_instructions`
(the field defaults to "No additional instructions.").

**On judge-LLM parsing failures**: `tunable_rag_evaluator` scores an item `0.0` any time it can't parse the judge
LLM's response into the expected `{score, reasoning}` JSON shape (e.g. the judge wrapped its answer in a markdown
code fence, added commentary before/after the JSON, or got truncated) — and that `0.0` looks *identical* in the
output to "the agent's answer was completely wrong." Those are very different failure modes, and conflating them
would give you a wrong read on both `nat eval` and, more importantly, the optimizer's fitness signal (a candidate
prompt could get penalized purely because the judge flaked on formatting, not because the prompt was worse).

**A worse, opposite failure mode**: if the react agent itself never reaches a `Final Answer` (e.g. it garbles a
tool-call name badly enough that it never recovers — small/weak models are especially prone to this), the workflow
returns an *empty* `generated_answer`. The judge should score that `0.0`, but our `judge_llm_prompt` originally
didn't say anything about handling empty input, and we observed it hallucinate a perfect `1.0` by treating the
*expected* answer as if it were the generated one — a false positive that's arguably worse than a parsing failure,
since it looks exactly like a genuinely correct run. We've since hardened `judge_llm_prompt` above to explicitly
require a `0.0` for empty answers, but the helper below still checks for this directly (rather than trusting the
judge to always comply), since a judge LLM ignoring an instruction is a "when," not an "if."

The next cell uses a small helper that flags both failure modes separately instead of silently folding them into
the average.

In [51]:
!nat eval --config_file optimizer_config.yml

2026-07-02 16:46:47 - INFO     - nat.plugins.eval.runtime.evaluate:723 - Starting evaluation run with config file: optimizer_config.yml
2026-07-02 16:46:47 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
Running workflow:  17%|████▎                     | 1/6 [00:01<00:06,  1.33s/it]2026-07-02 16:46:49 - INFO     - nat.plugins.langchain.agent.react_agent.agent:308 - [AGENT] Agent produced direct answer without ReAct format, accepting as final answer
2026-07-02 16:46:49 - INFO     - nat.plugins.langchain.agent.react_agent.agent:308 - [AGENT] Agent produced direct answer without ReAct format, accepting as final answer
Running workflow:  50%|█████████████             | 3/6 [00:01<00:01,  2.17it/s]2026-07-02 16:46:49 - INFO     - nat.plugins.langchain.agent.react_agent.agent:308 - [AGENT] Agent produced direct answer without ReAct format, accepting as final answer
2026-07-02 16:46:49 - INFO     - nat.plugins.langchain.agent.react_agent.agent:308 - [AGENT] 

In [52]:
import json


def summarize_eval_output(path, label):
    """Report the evaluator's average score, but separate out two distinct failure modes that
    both hide inside a normal-looking score:

    1. Judge-LLM parsing failures: TunableRagEvaluator scores an item 0.0 whenever it can't parse
       the judge's response — indistinguishable from a genuine "wrong answer" in average_score.
    2. Empty generated_answer: the workflow itself returned nothing (e.g. the agent got stuck on a
       malformed tool call and never reached a Final Answer). The judge should score this 0.0, but
       won't reliably do so unless told to — it can hallucinate a score by treating the *expected*
       answer as if it were the generated one. This is worse than case 1: it produces a false
       *positive* (a perfect-looking score for an answer that was never actually given) instead of
       a false negative.

    Both get excluded from the adjusted average, with a warning naming which item IDs and why.
    """
    with open(path) as f:
        data = json.load(f)

    parse_failures = []
    empty_answers = []
    judged_scores = []
    for item in data["eval_output_items"]:
        reasoning = item.get("reasoning")
        inner_reasoning = reasoning.get("reasoning") if isinstance(reasoning, dict) else reasoning
        generated_answer = reasoning.get("generated_answer") if isinstance(reasoning, dict) else None
        if isinstance(inner_reasoning, str) and inner_reasoning.startswith("Error in evaluator"):
            parse_failures.append(item["id"])
        elif not generated_answer or not str(generated_answer).strip():
            empty_answers.append(item["id"])
        else:
            judged_scores.append(item["score"])

    raw_avg = data["average_score"]
    adjusted_avg = sum(judged_scores) / len(judged_scores) if judged_scores else None
    n = len(data["eval_output_items"])

    print(f"{label}: raw average (as reported by nat eval) = {raw_avg}")
    if parse_failures:
        print(f"  WARNING: {len(parse_failures)}/{n} item(s) failed judge-LLM response parsing "
              f"(ids={parse_failures}). Those were scored 0.0 by default — that reflects a "
              f"parsing failure, NOT necessarily a wrong answer.")
    if empty_answers:
        flagged_scores = {item["id"]: item["score"] for item in data["eval_output_items"] if item["id"] in empty_answers}
        print(f"  CRITICAL: {len(empty_answers)}/{n} item(s) had an EMPTY generated_answer (ids={empty_answers}) — "
              f"the workflow returned no answer at all (check workflow_output.json / the agent logs for why; a "
              f"common cause is the agent garbling a tool-call name and never recovering). The judge still scored "
              f"these ({flagged_scores}) instead of reliably assigning 0.0 — it may be hallucinating a score from "
              f"the expected answer rather than actually grading nothing. Excluded from the adjusted average.")
    if parse_failures or empty_answers:
        print(f"  Adjusted average over the {len(judged_scores)} reliably-judged item(s): "
              f"{adjusted_avg if adjusted_avg is not None else 'N/A — no item was reliably judged'}")

    return {
        "raw_avg": raw_avg,
        "adjusted_avg": adjusted_avg,
        "parse_failures": parse_failures,
        "empty_answers": empty_answers,
    }


baseline_accuracy = summarize_eval_output("eval_baseline/accuracy_output.json", "Baseline")

Baseline: raw average (as reported by nat eval) = 0.15


## 4. Run the optimizer

`nat optimize` will:
1. Seed a population of `ga_population_size` prompt candidates (the first is our original "No additional
   instructions." prompt; the rest are LLM-generated mutations of it).
2. Evaluate every candidate on `eval_dataset.json` using the `accuracy` evaluator.
3. Select, crossover, and mutate to produce the next generation.
4. Repeat for `ga_generations` generations, then write out the best prompt found.

With `ga_population_size: 6` and `ga_generations: 3`, this runs on the order of ~20 workflow executions per
generation (population size × dataset size), so expect this cell to take a few minutes depending on API latency.

**A note on the `optimizer_prompt` overrides above**: `react_agent` splices `additional_instructions` directly
into its system prompt as plain text (`prompt_str += f" {config.additional_instructions}"` in the toolkit's
`react_agent/agent.py`), and that whole string is then parsed as an f-string template by LangChain's
`ChatPromptTemplate`. If the GA's mutator LLM ever writes a literal `{` or `}` (e.g. because it decides to
illustrate a JSON schema), template parsing crashes with `ValueError: expected '}' before end of string` and
kills the entire `nat optimize` run, not just that one candidate. That's why the `prompt_mutator` /
`prompt_crossover` functions above explicitly forbid curly braces, code blocks, and JSON, and why `nim_llm` has a
modest `max_tokens: 200` cap (shorter mutations are less likely to wander into writing a schema). If you still hit
that error, it's most likely a one-off — GA mutation is stochastic, so simply re-running this cell will usually
avoid the same unlucky output.

In [53]:
!nat optimize --config_file optimizer_config.yml

2026-07-02 16:46:59 - WARNING  - nat.experimental.decorators.experimental_warning_decorator:59 - The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.plugins.config_optimizer.optimizer_runtime.optimize_config
2026-07-02 16:47:00 - INFO     - nat.plugins.config_optimizer.prompts.ga_prompt_optimizer:464 - GA Prompt optimization ready: init_fn=prompt_mutator, recombine_fn=prompt_crossover
2026-07-02 16:47:03 - INFO     - nat.plugins.config_optimizer.prompts.ga_prompt_optimizer:590 - [GA] Generation 1/2: evaluating population of 3
2026-07-02 16:47:03 - INFO     - nat.plugins.eval.runtime.evaluate:723 - Starting evaluation run with config file: general=GeneralConfig(use_uvloop=None, telemetry=TelemetryConfig(logging={}, tracing={}), per_user_workflow_timeout=datetime.timedelta(seconds=1800), per_user_workflow_cleanup_interval=datetime.timedelta(seconds=300), enable_per_user_monitoring=Fa

## 5. Inspect the results

`optimizer_results/` now contains:

- `optimized_prompts.json` — the best prompt set found
- `optimized_prompts_gen<N>.json` — the best prompt set after each generation
- `ga_history_prompts.csv` — per-individual fitness across every generation

Note: unlike what the upstream docs describe, this pinned `nvidia-nat==1.8.0` release does **not** write an
`optimized_config.yml`. Looking at the installed source (`nat/plugins/config_optimizer/cli/optimize.py`), the CLI's
`nat optimize` command calls `optimize_config(...)` and discards its return value — no config file is ever written
to disk for the GA-prompt-only path. So we'll build our own "optimized" config file in the next cell from
`optimized_prompts.json`.

Each entry in `optimized_prompts.json` is keyed by the dotted parameter path (e.g.
`"workflow.additional_instructions"`) and maps to a 2-element list: `[winning_prompt_text, prompt_purpose]`. The
second element is just the field's `prompt_purpose` description (metadata), not a second candidate — the prompt to
use is index `[0]`.

In [54]:
with open("optimizer_results/optimized_prompts.json") as f:
    optimized_prompts = json.load(f)

winning_instructions = optimized_prompts["workflow.additional_instructions"][0]
print("Original additional_instructions:\n  No additional instructions.\n")
print("Optimized additional_instructions:\n ", winning_instructions)

Original additional_instructions:
  No additional instructions.

Optimized additional_instructions:
  To optimize the given prompt, focus on clarity, precision, and clarity and precision. Remove vague language, strengthen directives, and order sections as requested. This can be achieved by breaking down the original prompt into its most basic components while preserving the objective and task. For instance, break down the substitution of "at least one accepting parameter" into separate sentences or key points, and then reassemble them into the combined prompt. Introduceithereby, enhanced consideration can be given to contextual and inferential aspects of the prompt. TakingIntoConsideration the fact that the modified prompt soughtto SurvivecriticalsectionspecificDNSiguresatuErrMsg on crusExam cant軵enariosorAnswershelupto anew prompttargetsavesystem другихsmtp socioeconomic schoolk協791depending donn sustained Partnerclick Hero alumцион Emerg wy Actuallyएसлся homemadejob Aspect live pass 

In [55]:
import pandas as pd

history = pd.read_csv("optimizer_results/ga_history_prompts.csv")
print(history.columns.tolist())
history.tail(10)

['generation', 'index', 'metric::accuracy', 'scalar_fitness']


,generation,index,metric::accuracy,scalar_fitness
0,1,0,0.03,1.000000e+00
1,1,1,0.00,1.000000e-12
2,1,2,0.00,1.000000e-12
3,2,0,0.00,1.000000e-12
4,2,1,0.17,9.444444e-01
5,2,2,0.18,1.000000e+00


In [56]:
import yaml

with open("optimizer_config.yml") as f:
    optimized_config = yaml.safe_load(f)

optimized_config["workflow"]["additional_instructions"] = winning_instructions

with open("optimizer_results/optimized_config.yml", "w") as f:
    yaml.safe_dump(optimized_config, f, sort_keys=False)

print("Wrote optimizer_results/optimized_config.yml with the winning additional_instructions.")

Wrote optimizer_results/optimized_config.yml with the winning additional_instructions.


## 6. Re-score with the optimized prompt

`optimizer_results/optimized_config.yml` (the one we just built) is a complete, ready-to-run workflow config — it's
a copy of `optimizer_config.yml` with `workflow.additional_instructions` swapped for the GA's winning prompt, so we
can run `nat eval` against it directly. We point the output to a fresh directory with `--override` so we don't
overwrite the baseline results.

**Why `--log-level warning` here**: whenever `nat` applies a config `--override`, it logs the *entire effective
config* at INFO level to confirm what changed (`nat/cli/cli_utils/config_override.py`) — including `api_key`
fields, in plain text. If you save this notebook with that INFO output still attached to the cell, your real API
key ends up committed to the file. Since the evaluation summary table below is printed via `click.echo` (not the
logger), raising the log level to `warning` suppresses the secret-bearing config dump without losing anything
useful. This is the only cell in these notebooks that uses `--override`, so it's the only one that needed this.

In [57]:
!nat --log-level warning eval --config_file optimizer_results/optimized_config.yml --override eval.general.output_dir ./eval_optimized

Evaluating: 100%|████████████████████████████████| 6/6 [00:05<00:00,  1.20it/s]

=== EVALUATION SUMMARY ===
Workflow Status: COMPLETED (workflow_output.json)
Total Runtime: 3.51s

Per evaluator results:
| Evaluator   |   Avg Score | Output File          |
|-------------|-------------|----------------------|
| accuracy    |         0.2 | accuracy_output.json |



In [58]:
optimized_accuracy = summarize_eval_output("eval_optimized/accuracy_output.json", "Optimized")

print()
print("Baseline raw average:      ", baseline_accuracy["raw_avg"])
print("Optimized raw average:     ", optimized_accuracy["raw_avg"])
print("Baseline adjusted average: ", baseline_accuracy["adjusted_avg"])
print("Optimized adjusted average:", optimized_accuracy["adjusted_avg"])
if (baseline_accuracy["parse_failures"] or optimized_accuracy["parse_failures"]
        or baseline_accuracy["empty_answers"] or optimized_accuracy["empty_answers"]):
    print("\nCompare the *adjusted* averages, not the raw ones — either run had at least one unreliable "
          "score (a judge parsing failure or an empty generated_answer), which would otherwise distort the "
          "raw average in either direction.")

Optimized: raw average (as reported by nat eval) = 0.2

Baseline raw average:       0.15
Optimized raw average:      0.2
Baseline adjusted average:  0.15
Optimized adjusted average: 0.19999999999999998


## Wrap-up

With a dataset this small and a search this short, don't expect a huge or perfectly stable improvement — the point
is to see the full loop working end to end. To get more out of this on a real workflow:

- Use a larger, more representative eval dataset (dozens–hundreds of examples).
- Increase `ga_population_size` / `ga_generations` (start around 10–16 / 5–8, per the toolkit's own tuning guide).
- Increase `reps_per_param_set` (3–5) if your workflow's output is stochastic, to reduce evaluation noise.
- Set `optimizer.numeric.enabled: true` to *also* tune numeric fields like `temperature`/`top_p` via Optuna,
  concurrently with the prompt GA.
- Try `oracle_feedback_mode: "adaptive"` under `optimizer.prompt` to feed the judge's failure reasoning back into
  prompt mutations instead of mutating blindly.

**A residual limitation on judge-parsing failures**: `summarize_eval_output` above can audit and correct for
parsing failures *after the fact* because we control that code — we load the JSON output ourselves. `nat optimize`
doesn't give us that hook: internally, each GA individual's fitness is computed straight from `nat eval`'s raw
scores (including any `0.0`s from parsing failures), so a candidate prompt can end up penalized purely because the
judge flaked on formatting for one of its evaluation runs, not because the prompt was actually worse. Hardening
`judge_llm_prompt` and using the simpler `default_scoring: false` schema (both done above) reduces how often this
happens, but can't guarantee zero — if the GA's per-generation scores in `ga_history_prompts.csv` look noisier than
you'd expect, that's the first thing to suspect. Increasing `reps_per_param_set` averages out some of this noise
per individual, at the cost of more evaluator calls.

See the [Optimizer docs](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/main/docs/source/improve-workflows/optimizer.md)
and [Evaluation docs](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/main/docs/source/improve-workflows/evaluate.md)
for the full set of options — but note that both, as upstream markdown, describe the toolkit's `main` branch and can
drift from whatever version you have pinned in `requirements.txt`. When something in those docs doesn't match what
you see (as happened twice in this notebook — the `ragas` evaluator and `optimized_config.yml`), the fastest way to
confirm what's actually true for your install is to check the source directly: `python -c "import nat; print(nat.__path__)"`
plus `nat info components` beats trusting docs against a pinned version.